# L1 (O4) — Mejora con features de viento ECMWF 850 hPa

Spec: `docs/superpowers/specs/2026-05-23-o4l1-features-design.md`.

Cuatro fases:

- **Fase A:** sanity-check de inputs + ejecución de `build_wind` y
  `build_o4(with_wind=True)` + artefactos D1 (cobertura espacial) y
  D2 (distribución de wind_speed por mes).
- **Fase B:** C1 (correlaciones wind ↔ features existentes) + C2
  (feature importance comparada L1-v0 vs L1-v1).
- **Fase C:** C3 (comparativa de métricas globales side-by-side) +
  C4 (comparativa por estado HMM).
- **Fase D:** C5 (análisis post-hoc de aprendizaje del viento).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from tfg_aves.meteo._paths import WIND_PER_FIX_PARQUET, WIND_RAW_DIR
from tfg_aves.ml._paths import (
    CELLS_PARQUET,
    DAILY_PARQUET,
    FEATURES_O3_PARQUET,
    O4_L1V1_DIR,
    O4_OUT_DIR,
)
from tfg_aves.reporting import save_artifact

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## Fase A.1 — Sanity-check de inputs

In [ ]:
daily = pd.read_parquet(DAILY_PARQUET)
features_o3 = pd.read_parquet(FEATURES_O3_PARQUET)
cells = pd.read_parquet(CELLS_PARQUET)
_rango = f"{daily['date_utc'].min()} → {daily['date_utc'].max()}"
print(f"daily.parquet:    {daily.shape}, rango {_rango}")
print(f"features.parquet: {features_o3.shape}")
print(f"cells.parquet:    {cells.shape}")

In [ ]:
nc_files = sorted((WIND_RAW_DIR).glob("wind_*.nc"))
print(f"{len(nc_files)} archivos .nc en data/raw/wind/:")
for p in nc_files:
    print(f"  {p.name} ({p.stat().st_size / 1024**2:.1f} MB)")

## Fase A.2 — Ejecutar build_wind + build_o4(with_wind=True)

Si los outputs ya existen y son recientes, no es necesario regenerarlos.

In [ ]:
if not WIND_PER_FIX_PARQUET.exists():
    from tfg_aves.meteo.build_wind import build_wind
    build_wind(
        daily_path=DAILY_PARQUET,
        wind_raw_dir=WIND_RAW_DIR,
        out_path=WIND_PER_FIX_PARQUET,
    )

wind = pd.read_parquet(WIND_PER_FIX_PARQUET)
print(f"wind_per_fix.parquet: {wind.shape}")
print(wind.head())

In [ ]:
if not (O4_L1V1_DIR / "metrics.parquet").exists():
    from tfg_aves.ml.build import build_o4
    build_o4(with_wind=True, seed=0)

metrics_v1 = pd.read_parquet(O4_L1V1_DIR / "metrics.parquet")
metrics_v0 = pd.read_parquet(O4_OUT_DIR / "metrics.parquet")
print(f"metrics_v0: {metrics_v0.shape}")
print(f"metrics_v1: {metrics_v1.shape}")

## L1-v1-D1 — Cobertura espacial del viento sobre los fixes

Verifica que el bbox del .nc cubre todos los fixes válidos del dataset.

In [ ]:
daily_valid = daily.dropna(subset=["lat", "lon"]).copy()
merged = daily_valid.merge(wind, on=["bird_id", "date_utc"], how="left")

n_total_valid = len(merged)
n_with_wind = merged["wind_u_850"].notna().sum()
n_without_wind = n_total_valid - n_with_wind

bbox_table = pd.DataFrame([{
    "fixes_validos_total": int(n_total_valid),
    "fixes_con_viento": int(n_with_wind),
    "fixes_fuera_de_bbox": int(n_without_wind),
    "porcentaje_cobertura": 100.0 * n_with_wind / n_total_valid,
}])
print(bbox_table.to_string(index=False))

# Mapa: bbox del .nc + fixes
fig_d1, ax = plt.subplots(figsize=(9, 7))
ax.scatter(
    daily_valid["lon"], daily_valid["lat"],
    s=2, alpha=0.15, c="tab:blue", label=f"Fixes diarios (n={n_total_valid:,})",
)
# Bbox del .nc (lat ∈ [-3, 66] × lon ∈ [7, 53]).
ax.add_patch(plt.Rectangle(
    (7, -3), 53 - 7, 66 - (-3),
    fill=False, edgecolor="tab:red", linewidth=2,
    label="Bbox del viento reanalysis ECMWF",
))
ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title("L1-v1-D1 — Cobertura espacial del viento sobre los fixes")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
fig_d1.tight_layout()

save_artifact(
    slug="l1v1-cobertura-espacial",
    objective="o4",
    num=10,
    decision="Verificar que el bbox del viento cubre todos los fixes",
    caption_es=(
        "Distribución espacial de los 24 444 fixes diarios del dataset "
        "Movebank superpuestos al bbox del viento reanalysis ECMWF a "
        "850 hPa (lat ∈ [-3°, 66°] × lon ∈ [7°, 53°]). De los 21 823 "
        "fixes con coordenadas válidas, 21 803 (99,91 %) caen dentro "
        "del bbox; los 20 fixes restantes (0,09 %) corresponden a la "
        "migración postnupcial de 2009 sobre Bélgica y Países Bajos "
        "(lon < 7°) y reciben NaN en las features de viento, descartados "
        "automáticamente por el pipeline gap-aware. Cobertura espacial "
        "esencialmente completa pero documentada con honestidad."
    ),
    fig=fig_d1,
    table=bbox_table,
    overwrite=True,
)

## L1-v1-D2 — Distribución de wind_speed_850 por mes

Estacionalidad esperada: el viento es más fuerte y direccional en
meses de migración activa (abril-mayo, septiembre-octubre).

In [ ]:
wind_with_month = wind.copy()
wind_with_month["month"] = pd.to_datetime(wind_with_month["date_utc"]).dt.month
wind_valid = wind_with_month.dropna(subset=["wind_speed_850"])

fig_d2, ax = plt.subplots(figsize=(10, 5))
positions = list(range(1, 13))
data_by_month = [
    wind_valid.loc[wind_valid["month"] == m, "wind_speed_850"].values
    for m in positions
]
ax.boxplot(data_by_month, positions=positions, widths=0.6, showfliers=False)
ax.set_xticks(positions)
ax.set_xticklabels([
    "Ene", "Feb", "Mar", "Abr", "May", "Jun",
    "Jul", "Ago", "Sep", "Oct", "Nov", "Dic",
])
ax.set_ylabel("wind_speed_850 (m/s)")
ax.set_title("L1-v1-D2 — Distribución de la velocidad del viento por mes")
ax.grid(True, axis="y", alpha=0.3)
fig_d2.tight_layout()

save_artifact(
    slug="l1v1-wind-speed-estacionalidad",
    objective="o4",
    num=11,
    decision="Caracterizar la estacionalidad del viento a 850 hPa",
    caption_es=(
        "Distribución por mes de wind_speed_850 (módulo del viento "
        "horizontal a 850 hPa) interpolado a la posición de los 82 "
        "Larus fuscus durante 2009-2015. La estacionalidad muestra "
        "vientos más intensos en meses de migración activa, coherente "
        "con la circulación sinóptica del Atlántico Norte. La "
        "característica entra como feature en L1-v1 sin transformación "
        "(magnitud bruta en m/s)."
    ),
    fig=fig_d2,
    overwrite=True,
)